# Filter abstract prediction results

Input virus entities should include taxids (`name|taxid`) and disease entities should include MeSH IDs (`name|MESH:D...`).


In [ ]:
from pathlib import Path
import re
import pandas as pd

PREDICTIONS_TSV = Path("results/abstract_sentence_predictions.tsv")
OUTPUT_EVIDENCE_TSV = Path("results/abstract_virus_disease_evidence.tsv")
OUTPUT_PAIR_GROUPS_XLSX = Path("results/abstract_virus_disease_pair_groups.xlsx")

POSITIVE_PROB_THRESHOLD = 0.74
INPUT_COLUMNS = ["pmid", "virus", "disease", "sentence", "prob", "label"]
PAIR_GROUP_COLUMNS = ["pair_group_id", "virus", "virus_taxid", "disease", "mesh_id", "pmid", "sentence", "prob"]

BROAD_DISEASE_RE = re.compile(r"\b(infection|infections|disease|illness|cases?|deaths?|outbreak|epidemic|eruption|eruptions)\b", re.I)
NONWORD_RE = re.compile(r"[^\w]+")
BROAD_DISEASE_NAMES = {
    "infection", "infections", "viral infection", "viral infections",
    "disease", "illness", "outbreak", "epidemic", "death", "deaths",
}


In [ ]:
def split_entities(value):
    return [item.strip() for item in str(value).split(",") if item.strip()]


def parse_virus_entity(entity):
    parts = [part.strip() for part in str(entity).split("|")]
    if len(parts) < 2:
        return None
    taxid_text = parts[-1]
    if not taxid_text.isdigit():
        return None
    return parts[0], int(taxid_text)


def parse_disease_entity(entity):
    parts = [part.strip() for part in str(entity).split("|")]
    if len(parts) < 2:
        return None
    mesh_id = parts[-1].split(":")[-1].strip()
    if not mesh_id.startswith("D"):
        return None
    return parts[0], mesh_id


def is_specific_disease_name(name):
    normalized = NONWORD_RE.sub(" ", str(name)).strip()
    if normalized == "":
        return False
    if BROAD_DISEASE_RE.fullmatch(normalized):
        return False
    if normalized.lower() in BROAD_DISEASE_NAMES:
        return False
    return True


def read_prediction_table(path):
    df = pd.read_csv(path, sep="\t", header=None, names=INPUT_COLUMNS)
    df["prob"] = pd.to_numeric(df["prob"], errors="coerce")
    df["label"] = pd.to_numeric(df["label"], errors="coerce")
    df = df[(df["label"] == 1) & (df["prob"] >= POSITIVE_PROB_THRESHOLD)].copy()
    return df.dropna(subset=["prob"])


def expand_candidate_pairs(df):
    rows = []
    for row in df.itertuples(index=False):
        viruses = [parse_virus_entity(v) for v in split_entities(row.virus)]
        diseases = [parse_disease_entity(d) for d in split_entities(row.disease)]
        viruses = [v for v in viruses if v is not None]
        diseases = [d for d in diseases if d is not None and is_specific_disease_name(d[0])]
        for virus_name, virus_taxid in viruses:
            for disease_name, mesh_id in diseases:
                rows.append({
                    "pmid": str(row.pmid),
                    "virus": virus_name,
                    "virus_taxid": virus_taxid,
                    "disease": disease_name,
                    "mesh_id": mesh_id,
                    "sentence": str(row.sentence).replace("\n", " "),
                    "prob": row.prob,
                })
    expanded = pd.DataFrame(rows)
    if expanded.empty:
        return expanded
    expanded["pair_key"] = expanded["virus_taxid"].astype(str) + "|" + expanded["mesh_id"]
    return expanded


def keep_best_evidence_per_article_pair(df):
    if df.empty:
        return df
    df = df.sort_values(["pmid", "virus_taxid", "mesh_id", "prob"], ascending=[True, True, True, False])
    best_idx = df.groupby(["pmid", "virus_taxid", "mesh_id"], sort=False)["prob"].idxmax()
    return df.loc[best_idx].sort_values(["virus", "disease", "pmid"], kind="stable").reset_index(drop=True)


def build_pair_groups(evidence):
    rows = []
    if evidence.empty:
        return pd.DataFrame(columns=PAIR_GROUP_COLUMNS)
    pair_ids = {pair: i + 1 for i, pair in enumerate(evidence["pair_key"].drop_duplicates())}
    for pair_key, group in evidence.groupby("pair_key", sort=False):
        group_id = pair_ids[pair_key]
        for i, row in enumerate(group.itertuples(index=False)):
            rows.append({
                "pair_group_id": group_id,
                "virus": row.virus if i == 0 else "",
                "virus_taxid": row.virus_taxid if i == 0 else "",
                "disease": row.disease if i == 0 else "",
                "mesh_id": row.mesh_id if i == 0 else "",
                "pmid": row.pmid,
                "sentence": row.sentence,
                "prob": row.prob,
            })
    return pd.DataFrame(rows, columns=PAIR_GROUP_COLUMNS)


In [ ]:
predictions = read_prediction_table(PREDICTIONS_TSV)
expanded_pairs = expand_candidate_pairs(predictions)
evidence = keep_best_evidence_per_article_pair(expanded_pairs)
pair_groups = build_pair_groups(evidence)

OUTPUT_EVIDENCE_TSV.parent.mkdir(parents=True, exist_ok=True)
evidence.to_csv(OUTPUT_EVIDENCE_TSV, sep="\t", index=False)
pair_groups.to_excel(OUTPUT_PAIR_GROUPS_XLSX, index=False)

print(f"positive sentence rows: {len(predictions)}")
print(f"expanded evidence rows: {len(expanded_pairs)}")
print(f"deduplicated evidence rows: {len(evidence)}")
print(f"virus-disease pairs: {pair_groups['pair_group_id'].nunique() if not pair_groups.empty else 0}")
